# AI Recipe Generator — Model Training Notebook

This notebook trains the LSTM-based recipe generation model.

**Fixes applied vs original:**
- ✅ `fit_generator()` → `model.fit()` (modern TensorFlow API)
- ✅ `Dense(total_words/2)` → `Dense(int(total_words/2))` (Python 3 fix)
- ✅ `max_sequence_length` capped at 200 to match `app.py`
- ✅ Tokenizer saved as `tokenizer.pkl`
- ✅ Config saved as `model_config.pkl`
- ✅ Model saved in modern `.keras` format
- ✅ GPU auto-detection added
- ✅ Early stopping added to prevent overfitting

**Instructions:**
1. Place `dataset.csv` in the same folder as this notebook
2. Run all cells top-to-bottom
3. Three files will be saved: `recipe_generation_model.keras`, `tokenizer.pkl`, `model_config.pkl`
4. Copy those three files to the project root before running `app.py`

In [ ]:
# Cell 1 — Imports
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

# GPU Detection
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU available: {gpus}')
else:
    print('No GPU found — running on CPU')

In [ ]:
# Cell 2 — Load Dataset
data = pd.read_csv('dataset.csv')
data.drop_duplicates(subset=['TranslatedInstructions'], inplace=True)
data.dropna(subset=['TranslatedInstructions'], inplace=True)
data = data[data['TranslatedInstructions'].str.strip() != '']
print(f'Loaded {len(data)} unique recipes')
data[['TranslatedRecipeName', 'TranslatedInstructions']].head(3)

In [ ]:
# Cell 3 — Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['TranslatedInstructions'])
total_words = len(tokenizer.word_index) + 1
print(f'Vocabulary size: {total_words}')

In [ ]:
# Cell 4 — Create N-gram Sequences
input_sequences = []
for line in data['TranslatedInstructions']:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)

print(f'Total sequences: {len(input_sequences)}')

# Cap max_sequence_length at 200 to match app.py inference
MAX_SEQ_LEN_CAP = 200
raw_max = max(len(seq) for seq in input_sequences)
max_sequence_length = min(raw_max, MAX_SEQ_LEN_CAP)
print(f'max_sequence_length (capped at {MAX_SEQ_LEN_CAP}): {max_sequence_length}')

In [ ]:
# Cell 5 — Pad Sequences & Prepare X, y
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_length, padding='pre')
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

In [ ]:
# Cell 6 — Build Model
model = Sequential([
    Embedding(total_words, 100, input_length=max_sequence_length - 1),
    Bidirectional(LSTM(150, return_sequences=True)),
    Dropout(0.2),
    LSTM(100),
    Dense(int(total_words / 2), activation='relu'),   # FIX: int() cast required
    Dense(total_words, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Cell 7 — Train Model
# FIX: model.fit() replaces deprecated model.fit_generator()
callbacks = [
    ModelCheckpoint('recipe_generation_model.keras', save_best_only=True, monitor='loss', verbose=1),
    EarlyStopping(monitor='loss', patience=5, restore_best_weights=True, verbose=1)
]

history = model.fit(
    X, y,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Cell 8 — Save Model, Tokenizer, Config

# Save model in modern Keras format (not legacy .h5)
model.save('recipe_generation_model.keras')
print('Model saved → recipe_generation_model.keras')

# Save tokenizer so app.py uses the EXACT same word-index mapping
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print('Tokenizer saved → tokenizer.pkl')

# Save config so app.py uses the EXACT same max_sequence_length
config = {
    'max_sequence_length': max_sequence_length,
    'total_words': total_words
}
with open('model_config.pkl', 'wb') as f:
    pickle.dump(config, f)
print('Config saved → model_config.pkl')

print('\n✅ All files saved. Copy them to the project root before running app.py')

In [ ]:
# Cell 9 — Plot Training History
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'])
ax1.set_title('Model Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'], color='orange')
ax2.set_title('Model Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100)
plt.show()
print('Plot saved → training_history.png')

In [ ]:
# Cell 10 — Test Generation
def generate_recipe(seed_text, next_words=15):
    seed_text = seed_text.strip().lower()
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_length - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)
        output_word = ''
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += ' ' + output_word
    return seed_text

# Try a few seeds
seeds = ['Heat oil in', 'Blend tomatoes garlic', 'Boil water add']
for seed in seeds:
    print(f'Seed: "{seed}"')
    print(f'Recipe: {generate_recipe(seed)}\n')